In [0]:
# ===================================================================
# Full Environment Cleanup Script
# Run this before executing the Job A and Job B test scripts.
# ===================================================================

# --- Configuration ---
target_table_name = "mq_gmdf_dp_poc.kafka_ewm_poc.ewm_ddl_ext"
control_table_schema = "mq_gmdf_dp_poc.metadata_ddl"
checkpoint_location_root = "/tmp/mq_job_b_checkpoints/"

# --- Table and Schema Definitions ---
catalog, target_db, target_tbl = target_table_name.split('.')
job_a_log_table = f"{control_table_schema}.job_a_execution_log"
stream_log_table = f"{control_table_schema}.stream_job_execution_log"
microbatch_log_table = f"{control_table_schema}.microbatch_execution_log"
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
checkpoint_path = f"{checkpoint_location_root.rstrip('/')}/{target_tbl}_checkpoint"

# --- Execution ---

print("Starting full environment cleanup...")

# 1. Drop the main target table
print(f"Dropping target table: {target_table_name}...")
spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")

# 2. Drop all control and log tables
print(f"Dropping control and log tables in schema: {control_table_schema}...")
spark.sql(f"DROP TABLE IF EXISTS {job_a_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {stream_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {microbatch_log_table}")
spark.sql(f"DROP TABLE IF EXISTS {schema_transition_table}")
spark.sql(f"DROP TABLE IF EXISTS {schema_store_columns_table}")

# 3. Drop the control schema itself
print(f"Dropping control schema: {control_table_schema}...")
spark.sql(f"DROP SCHEMA IF EXISTS {control_table_schema} CASCADE")

# 4. Remove the streaming checkpoint directory from storage
print(f"Removing checkpoint directory: {checkpoint_path}...")
try:
    dbutils.fs.rm(checkpoint_path, recurse=True)
    print("  - Checkpoint directory removed.")
except Exception as e:
    if "No such file or directory" in str(e):
        print("  - Checkpoint directory did not exist. Nothing to remove.")
    else:
        print(f"  - Warning: Failed to remove checkpoint directory. Error: {e}")

print("\n✅ Cleanup Complete. The environment is now clean.")

In [0]:
# ===================================================================================
#
# JOB A: UNIFIED SCHEMA ACTIVATOR & BACKFILL ORCHESTRATOR
# INTERACTIVE TEST SCRIPT (v12.4 - Definitive with Backfill & Naming Fix)
#
# ===================================================================================

import requests
import json
import hashlib
import uuid
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from datetime import datetime

# ===================================================================================
# --- CONFIGURATION ---
# ===================================================================================
ddl_url = "https://raw.githubusercontent.com/naveenbanappa/mq-dia-dbx-ewm-framework/main/schemas/ewm_ddl_ext.json"
control_table_schema = "mq_gmdf_dp_poc.metadata_ddl"
source_data_path = "abfss://topics@mqosidevadls01.dfs.core.windows.net/ZMW_I_INTFLOGS_v2/"
unique_id_column = "run_id"
triggering_event = "interactive_test"
# ===================================================================================

### Section 1: Initialization & Full Control Plane Bootstrapping ###
run_id = str(uuid.uuid4())
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"
job_a_execution_log_table = f"{control_table_schema}.job_a_execution_log"
stream_log_table = f"{control_table_schema}.stream_job_execution_log"
microbatch_log_table = f"{control_table_schema}.microbatch_execution_log"

print("✅ Job A Configuration Initialized")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {control_table_schema}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {job_a_execution_log_table} (run_id STRING, run_timestamp TIMESTAMP, status STRING, ddl_location STRING, triggering_event STRING, old_schema_hash STRING, new_schema_hash STRING, message STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema_transition_table} (target_table_name STRING, schema_hash STRING, ddl_location STRING, event_type STRING, event_ts TIMESTAMP, backfill_status STRING) USING DELTA PARTITIONED BY (target_table_name)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {schema_store_columns_table} (target_table_name STRING, contract_hash STRING, name_in_ddl STRING, physical_column_name STRING, type STRING, source_kind STRING, source_column STRING, json_key STRING, ddl_location STRING, created_at TIMESTAMP) USING DELTA PARTITIONED BY (target_table_name)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {stream_log_table} (run_id STRING, r_src_id STRING, etl_job STRING, table_name STRING, start_tmstmp TIMESTAMP, end_tmstmp TIMESTAMP, load_status_cd STRING, scheduling STRING, log_link STRING, error_details STRING) USING DELTA")
spark.sql(f"CREATE TABLE IF NOT EXISTS {microbatch_log_table} (microbatch_id STRING, run_id STRING, r_src_id STRING, etl_job STRING, table_name STRING, spark_batch_id LONG, start_tmstmp TIMESTAMP, end_tmstmp TIMESTAMP, load_status_cd STRING, schema_hash STRING, rows_ins_nbr LONG, error_details STRING) USING DELTA")
print("   - All control tables created or verified.")


### Section 2: Core Utility Functions ###

def fetch_ddl_from_url(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def calculate_canonical_hash(ddl_json: dict) -> str:
    hash_inputs = sorted([f"{c['name']};{c['type']};{c['source_mapping']}" for c in ddl_json.get("columns", [])])
    return hashlib.md5("||".join(hash_inputs).encode("utf-8")).hexdigest()

def get_latest_active_hash(table_name: str, target_table_filter: str) -> str:
    if not spark.catalog.tableExists(table_name): return ""
    query = f"SELECT schema_hash FROM {table_name} WHERE target_table_name = '{target_table_filter}' AND event_type = 'ACTIVE' LIMIT 1"
    result = spark.sql(query).collect()
    return result[0]["schema_hash"] if result else ""

def sync_delta_schema(target_table: str, processed_ddl_df: F.DataFrame):
    all_physical_columns = processed_ddl_df.collect()
    try:
        spark.table(target_table).schema
    except AnalysisException as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
            print(f"   - Table not found. Creating new table '{target_table}' from DDL...")
            columns_sql = [f"`{col['physical_column_name']}` {col['type'].upper()}" for col in all_physical_columns]
            if "_rescued_data" not in [c['physical_column_name'].lower() for c in all_physical_columns]:
                 columns_sql.append("`_rescued_data` STRING")
            spark.sql(f"CREATE TABLE {target_table} ({', '.join(columns_sql)}) USING DELTA")
            print("   - Table created successfully from DDL.")
            return
        else: raise e
    
    existing_columns_lower = {col.name.lower() for col in spark.table(target_table).schema.fields}
    new_columns_to_add = [col for col in all_physical_columns if col['physical_column_name'].lower() not in existing_columns_lower]
    if new_columns_to_add:
        add_cols_sql = [f"`{col['physical_column_name']}` {col['type'].upper()}" for col in new_columns_to_add]
        spark.sql(f"ALTER TABLE {target_table} ADD COLUMNS ({', '.join(add_cols_sql)})")

def update_job_a_log(status: str, message: str, old_hash: str = "", new_hash: str = ""):
    clean_message = message.replace("'", "''")
    spark.sql(f"MERGE INTO {job_a_execution_log_table} t USING (SELECT '{run_id}' as run_id) s ON t.run_id = s.run_id WHEN MATCHED THEN UPDATE SET status = '{status}', message = '{clean_message}', old_schema_hash = '{old_hash}', new_schema_hash = '{new_hash}' WHEN NOT MATCHED THEN INSERT (run_id, run_timestamp, status, ddl_location, triggering_event, message) VALUES ('{run_id}', current_timestamp(), '{status}', '{ddl_url}', '{triggering_event}', '{clean_message}')")
    print(f"   - Log updated for run {run_id} with status: {status}")

### Section 3: Main Processing & Backfill Logic ###
try:
    update_job_a_log("RUNNING", "Job started.")
    ddl_data = fetch_ddl_from_url(ddl_url)
    full_target_table = ddl_data.get("table_name")
    if not full_target_table: raise ValueError("DDL must contain 'table_name'.")
    
    simple_table_name = full_target_table.split('.')[-1]
    
    new_hash = calculate_canonical_hash(ddl_data)
    
    columns_data = []
    for col in ddl_data.get("columns", []):
        if mapping := col.get("source_mapping"):
            parts = mapping.split('.', 2)
            source_kind, source_column, json_key = parts[0].upper(), None, None
            if source_kind == 'JSON':
                if len(parts) < 3: raise ValueError(f"Invalid JSON mapping: {mapping}")
                source_column, json_key = parts[1], parts[2]
            elif len(parts) > 1: source_column = parts[1]
            columns_data.append((col['name'], col['name'], col['type'], source_kind, source_column, json_key))
    
    processed_ddl_df = spark.createDataFrame(columns_data, "name_in_ddl STRING, physical_column_name STRING, type STRING, source_kind STRING, source_column STRING, json_key STRING")
    latest_known_hash = get_latest_active_hash(schema_transition_table, simple_table_name)

    if new_hash == latest_known_hash:
        update_job_a_log("SUCCESS", "No schema change.", old_hash=latest_known_hash, new_hash=new_hash)
    else:
        old_schema_hash = latest_known_hash
        sync_delta_schema(full_target_table, processed_ddl_df)
        
        blueprint_df = processed_ddl_df.withColumn("target_table_name", F.lit(simple_table_name)).withColumn("contract_hash", F.lit(new_hash)).withColumn("ddl_location", F.lit(ddl_url)).withColumn("created_at", F.current_timestamp())
        blueprint_df.write.format("delta").mode("overwrite").option("replaceWhere", f"target_table_name = '{simple_table_name}' AND contract_hash = '{new_hash}'").saveAsTable(schema_store_columns_table)

        if old_schema_hash:
            print("\n   - Beginning backfill process...")
            spark.sql(f"UPDATE {schema_transition_table} SET event_type = 'ARCHIVED' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{old_schema_hash}'")
            spark.sql(f"INSERT INTO {schema_transition_table} VALUES ('{simple_table_name}', '{new_hash}', '{ddl_url}', 'ACTIVE', current_timestamp(), 'BACKFILL_IN_PROGRESS')")

            schema_store_df = spark.table(schema_store_columns_table).filter(f"target_table_name = '{simple_table_name}'")
            new_schema_bp = schema_store_df.filter(f"contract_hash = '{new_hash}'")
            old_schema_bp = schema_store_df.filter(f"contract_hash = '{old_schema_hash}'")
            new_cols = new_schema_bp.join(old_schema_bp, on="physical_column_name", how="left_anti").filter(~F.col("source_kind").startswith("PIPELINE")).collect()
            
            if not new_cols:
                spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
            else:
                rows_to_backfill_df = spark.table(full_target_table).filter(f"schema_hash_at_ingest = '{old_schema_hash}'")
                if rows_to_backfill_df.count() == 0:
                     spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
                else:
                    rows_to_backfill_df.select(unique_id_column).distinct().createOrReplaceTempView("keys_to_backfill")
                    source_df = spark.read.format("parquet").load(source_data_path)
                    backfill_source_df = source_df.join(spark.table("keys_to_backfill"), unique_id_column, "inner")
                    
                    processed_backfill_df = backfill_source_df
                    json_cols_to_process = [r for r in new_schema_bp.collect() if r['source_kind'] == 'JSON']
                    for r in json_cols_to_process:
                        processed_backfill_df = processed_backfill_df.withColumn(r['physical_column_name'], F.get_json_object(F.col(r['source_column']), f"$.{r['json_key']}"))
                    
                    final_cols_for_merge = [unique_id_column] + [c['physical_column_name'] for c in new_cols]
                    backfill_update_view = processed_backfill_df.select(final_cols_for_merge)
                    backfill_update_view.createOrReplaceTempView("backfill_source_view")
                    
                    update_clauses = [f"target.`{c['physical_column_name']}` = source.`{c['physical_column_name']}`" for c in new_cols]
                    update_clauses.append(f"target.schema_hash_at_ingest = '{new_hash}'")
                    
                    merge_sql = f"MERGE INTO {full_target_table} AS target USING backfill_source_view AS source ON target.{unique_id_column} = source.{unique_id_column} WHEN MATCHED AND target.schema_hash_at_ingest = '{old_schema_hash}' THEN UPDATE SET {', '.join(update_clauses)}"
                    spark.sql(merge_sql)
                    spark.sql(f"UPDATE {schema_transition_table} SET backfill_status = 'COMPLETE' WHERE target_table_name = '{simple_table_name}' AND schema_hash = '{new_hash}'")
        else:
            spark.sql(f"INSERT INTO {schema_transition_table} VALUES ('{simple_table_name}', '{new_hash}', '{ddl_url}', 'ACTIVE', current_timestamp(), 'NOT_APPLICABLE')")
            
        update_job_a_log("SUCCESS", "Schema activated.", old_hash=old_schema_hash, new_hash=new_hash)
except Exception as e:
    import traceback
    error_message = f"FATAL ERROR in Job A run {run_id}: {traceback.format_exc()}"
    update_job_a_log("FAIL", error_message)
    raise e

In [0]:
%sql
--select * from mq_gmdf_dp_poc.metadata_ddl.schema_transition
--select * from mq_gmdf_dp_poc.metadata_ddl.schema_store_columns
--select * frommq_gmdf_dp_poc.kafka_ewm_poc.ewm_ddl_ext

In [0]:
# ===================================================================================
#
# JOB A - POST-ACTIVATION VALIDATION SCRIPT (INTERACTIVE)
# v2.0 - Aligned with Simple Naming Convention
#
# ===================================================================================

from pyspark.sql import functions as F

# ===================================================================================
# --- CONFIGURATION FOR INTERACTIVE TEST ---
# --- ⬇️ **UPDATE THE HASH VALUE BEFORE RUNNING** ⬇️ ---
# ===================================================================================

# 1. Target Table Name (full name from DDL)
target_table_name_full = "mq_gmdf_dp_poc.kafka_ewm_poc.ewm_ddl_ext"

# 2. Control Table Schema
control_table_schema = "mq_gmdf_dp_poc.metadata_ddl"

# 3. Newly Activated Hash:
#    **COPY THIS VALUE** from the output of the main Job A script you just ran.
newly_activated_hash = "8eeb2b57a31c191ff54760d7d950dc56" # <-- IMPORTANT

# ===================================================================================
# --- SCRIPT EXECUTION STARTS HERE ---
# ===================================================================================

### Section 1: Initialization ###

if not all([target_table_name_full, control_table_schema, newly_activated_hash]) or "PASTE" in newly_activated_hash:
    raise ValueError("FATAL: Please update the configuration variables, especially 'newly_activated_hash', before running.")

# --- Table Names ---
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"

# ===================== CRITICAL FIX IS HERE =====================
# Use the simple table name for all metadata lookups, matching Job A and Job B
simple_table_name = target_table_name_full.split('.')[-1]
# ================================================================

print(f"✅ Starting Post-Activation Validation for:")
print(f"   - Target Table: {target_table_name_full}")
print(f"   - Simple Name Key: {simple_table_name}")
print(f"   - Hash to Validate: {newly_activated_hash}")


### Section 2: Validation Checks ###

failures = []

# --- Check 1: State Coherency in 'schema_transition' table ---
print(f"\n🔄 Running Check 1: State Coherency in 'schema_transition' (for key: '{simple_table_name}')...")
try:
    # Use simple_table_name in the filter
    transition_df = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}'")
    active_rows = transition_df.filter("event_type = 'ACTIVE'").collect()

    if len(active_rows) != 1:
        failures.append(f"FAILURE: Expected exactly 1 'ACTIVE' schema, but found {len(active_rows)}.")
    else:
        if active_rows[0]["schema_hash"] != newly_activated_hash:
            failures.append(f"FAILURE: The 'ACTIVE' schema hash is '{active_rows[0]['schema_hash']}', but expected '{newly_activated_hash}'.")
        else:
            print("  - PASSED: Exactly one schema is 'ACTIVE' and its hash is correct.")

    non_archived_rows = transition_df.filter("event_type NOT IN ('ACTIVE', 'ARCHIVED')").count()
    if non_archived_rows > 0:
        failures.append(f"FAILURE: Found {non_archived_rows} rows that were not 'ACTIVE' or 'ARCHIVED'. All old schemas must be archived.")
    else:
        print("  - PASSED: All non-active schemas are correctly 'ARCHIVED'.")

except Exception as e:
    failures.append(f"CRITICAL FAILURE during State Coherency check: {e}")

# --- Check 2: The "Backfill Completeness" Check ---
print(f"\n🔄 Running Check 2: Backfill Completeness in 'schema_transition' (for key: '{simple_table_name}')...")
try:
    # Use simple_table_name in the filter
    backfill_status_row = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}' AND schema_hash = '{newly_activated_hash}'").first()
    if not backfill_status_row:
        failures.append(f"FAILURE: Cannot find the transaction row for the new hash '{newly_activated_hash}'.")
    else:
        status = backfill_status_row["backfill_status"]
        if status not in ["COMPLETE", "NOT_APPLICABLE"]:
            failures.append(f"FAILURE: Expected backfill status to be 'COMPLETE' or 'NOT_APPLICABLE', but found '{status}'.")
        else:
            print(f"  - PASSED: Backfill status is correctly set to '{status}'.")
except Exception as e:
    failures.append(f"CRITICAL FAILURE during Backfill Completeness check: {e}")

# --- Check 3: The "Blueprint Existence" Check ---
print(f"\n🔄 Running Check 3: Blueprint Existence in 'schema_store_columns' (for key: '{simple_table_name}')...")
try:
    # Use simple_table_name in the filter
    blueprint_count = spark.table(schema_store_columns_table).filter(f"target_table_name = '{simple_table_name}' AND contract_hash = '{newly_activated_hash}'").count()
    if blueprint_count == 0:
        failures.append(f"FAILURE: No column blueprint found in '{schema_store_columns_table}' for the new hash '{newly_activated_hash}'. Job B will fail.")
    else:
        print(f"  - PASSED: Found {blueprint_count} blueprint columns for the new hash.")
except Exception as e:
    failures.append(f"CRITICAL FAILURE during Blueprint Existence check: {e}")


### Section 3: Final Verdict ###

if failures:
    print("\n\n❌ VALIDATION FAILED. The system may be in an inconsistent state.")
    for f in failures:
        print(f"  - {f}")
    raise Exception("Post-activation validation checks failed. Please review the errors.")
else:
    success_msg = "✅ All post-activation validation checks passed successfully."
    print(f"\n\n{success_msg}")